# 第 1 章 Notebook：Agent Harness 与最小运行结构

对应课程章节：[第 1 章：从 Agent Framework 到 Agent Harness — Deep Agents 的诞生逻辑](../../content/ch01-agent-harness.md)

本 Notebook 用一个最小示例，把第 1 章讲的「Agent 三层架构」跑起来：先分别用 LangChain 的 `create_agent()` 和 Deep Agents 的 `create_deep_agent()` 创建最小 Agent，注册同一个工具，观察两者默认装配能力的差异，并查看一次完整 `invoke` 返回的中间状态。

## 学习目标

跑完本 Notebook 后，你应该能够：

1. 说明 LangGraph（Runtime）、LangChain（Framework）、Deep Agents（Harness）三层各自负责什么；
2. 用 `create_agent()` 创建一个最小 LangChain Agent，并注册一个本地工具；
3. 用 `create_deep_agent()` 创建一个最小 Deep Agent，对比它与前者默认工具集的差异；
4. 阅读一次 `invoke` 返回的 `messages` 列表，识别用户消息、模型消息和工具消息；
5. 通过 `MODEL_NAME` 和 API Key 环境变量配置模型，而不把密钥写进代码。

对应课程小节：

- 「Agent 开发的三个层次」：底层 Runtime / 中间层 Framework / 上层 Harness；
- 「Deep Agents 的核心设计理念」：Harness 预置的虚拟文件系统等工具；
- 「三层关系一览」：三层不是替代关系，而是自底向上层层构建。

## 运行环境与依赖

本 Notebook 在以下环境验证：

| 项目 | 版本 |
|---|---|
| 操作系统 | macOS（Apple Silicon） |
| Python | 3.12 |
| deepagents | 0.7.15 |
| langchain | 1.4.2 |
| langgraph | 1.2.11 |
| langchain-openai | 1.6.2 |

支持平台：macOS、Linux 与 WSL（Windows 原生未验证）。安装命令（推荐使用 `uv`，Python 要求 3.11+）：

```bash
uv venv --python 3.12
source .venv/bin/activate
uv pip install "deepagents>=0.7,<0.8" langchain langgraph langchain-openai python-dotenv
```

若要在浏览器外验证，另需 `ipykernel` 与 `nbconvert`。

## 环境变量与外部服务

本 Notebook 需要一个可用的 LLM API。课程默认使用 [硅基流动（SiliconFlow）](https://siliconflow.cn/) 作为模型提供商（兼容 OpenAI 接口），相关环境变量：

| 变量 | 是否必需 | 说明 |
|---|---|---|
| `SILICONFLOW_API_KEY` | 必需 | 模型服务商密钥，请用自己的 Key |
| `MODEL_NAME` | 可选 | 模型名，默认 `Qwen/Qwen2.5-7B-Instruct` |
| `SILICONFLOW_BASE_URL` | 可选 | 接口地址，默认 `https://api.siliconflow.cn/v1` |

请通过环境变量或仓库根目录的 `.env` 文件提供密钥，**不要**把真实密钥写进 Notebook 或提交到 Git。`.env` 已加入 `.gitignore`。

仓库根目录提供了 `.env.example`，复制后填入自己的密钥即可：
在终端执行如下指令：

```bash
cp .env.example .env
# 然后编辑 .env，把 SILICONFLOW_API_KEY 换成自己的 Key
```

外部服务与成本：本示例只进行少量短对话，调用量很小；具体计费与免费额度以模型平台页面为准，需要一个可访问该接口的网络环境。

## 0. 准备：确认依赖版本

先确认环境里装好、且版本符合课程基线。预期输出（具体补丁版本可能随安装时间略有不同）：

```text
deepagents==0.7.15
langchain==1.4.2
langgraph==1.2.11
langchain-openai==1.6.2
```

如果某一行显示「未安装」，请回到前面的「运行环境与依赖」一节安装对应包。

In [1]:
import importlib.metadata as md

for name in ["deepagents", "langchain", "langgraph", "langchain-openai"]:
    try:
        print(f"{name}=={md.version(name)}")
    except md.PackageNotFoundError:
        print(f"{name}: 未安装")

deepagents==0.7.15
langchain==1.4.2
langgraph==1.2.11
langchain-openai==1.6.2


## 1. 读取模型配置并初始化模型

先把密钥和模型名从环境变量读出来。若缺少密钥，这里会给出明确报错，而不是让后续单元格莫名失败。

In [2]:
import os

from dotenv import load_dotenv

load_dotenv()  # 从仓库根目录的 .env 读取；没有该文件时会静默跳过

MODEL_NAME = os.environ.get("MODEL_NAME", "Qwen/Qwen2.5-7B-Instruct")
BASE_URL = os.environ.get("SILICONFLOW_BASE_URL", "https://api.siliconflow.cn/v1")
API_KEY = os.environ.get("SILICONFLOW_API_KEY")

print("MODEL_NAME =", MODEL_NAME)
print("BASE_URL   =", BASE_URL)
print("API_KEY    =", "已设置" if API_KEY else "未设置")

if not API_KEY:
    raise RuntimeError(
        "未检测到 SILICONFLOW_API_KEY。请先执行 "
        "`export SILICONFLOW_API_KEY=...`，或在本仓库根目录创建 .env 文件后重跑。"
    )

MODEL_NAME = Qwen/Qwen2.5-7B-Instruct
BASE_URL   = https://api.siliconflow.cn/v1
API_KEY    = 已设置


In [3]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model=MODEL_NAME,
    api_key=API_KEY,
    base_url=BASE_URL,
    temperature=0,
    timeout=60,
)

print("模型类型:", type(model).__name__)
print("模型名  :", model.model_name)
print("接口地址:", model.openai_api_base)

模型类型: ChatOpenAI
模型名  : Qwen/Qwen2.5-7B-Instruct
接口地址: https://api.siliconflow.cn/v1


## 2. 定义一个最简单的工具

工具是 Agent 能调用的外部能力。这里定义一个 `echo` 工具，输入什么就原样返回什么，便于观察「模型决定调用工具 → 工具执行 → 模型总结结果」这条链路。

In [4]:
from langchain_core.tools import tool


@tool
def echo(text: str) -> str:
    """原样返回传入的文本，用于演示 Agent 的工具调用。"""
    return f"echo: {text}"

## 3. 中间层：用 LangChain 的 `create_agent()` 创建最小 Agent

`create_agent()` 来自 LangChain 框架层。它只装配你显式传入的模型和工具——这是理解「框架层提供什么」的基线。

In [5]:
from langchain.agents import create_agent

langchain_agent = create_agent(
    model=model,
    tools=[echo],
    system_prompt="你是一个简洁的助手。需要工具时优先调用工具，再用一句话总结结果。",
)

print("图节点:", list(langchain_agent.get_graph().nodes))

图节点: ['__start__', 'model', 'tools', '__end__']


### 3.1 跑一次完整 invoke，观察中间状态

`invoke` 返回一个字典，`result["messages"]` 里按顺序记录了整条轨迹：`HumanMessage`（用户输入）→ `AIMessage`（可能带 `tool_calls`）→ `ToolMessage`（工具返回值）→ `AIMessage`（最终回答）。

In [6]:
question = "请调用 echo 工具，把文本 'hello deepagents' 原样返回，然后告诉我工具返回了什么。"

lc_result = langchain_agent.invoke(
    {"messages": [{"role": "user", "content": question}]}
)

for message in lc_result["messages"]:
    message.pretty_print()

================================ Human Message =================================

请调用 echo 工具，把文本 'hello deepagents' 原样返回，然后告诉我工具返回了什么。
================================== Ai Message ==================================
Tool Calls:
  echo (01a0bdaecd12fb6b0dbca6132f43eda0)
 Call ID: 01a0bdaecd12fb6b0dbca6132f43eda0
  Args:
    text: hello deepagents
================================= Tool Message =================================
Name: echo

echo: hello deepagents
================================== Ai Message ==================================

工具返回了文本 "hello deepagents"。


### 3.2 检查可验证的行为

模型措辞不固定，但「是否发生工具调用」「消息类型顺序」是稳定的。下面只对结构做断言，并在未触发工具调用时给出提示。

In [7]:
from langchain_core.messages import AIMessage, ToolMessage

lc_messages = lc_result["messages"]
lc_tool_messages = [m for m in lc_messages if isinstance(m, ToolMessage)]

assert isinstance(lc_messages[-1], AIMessage), "最后一条消息应当是模型的最终回复"

if lc_tool_messages:
    print("✓ 检测到工具调用，工具返回:", lc_tool_messages[-1].content)
else:
    print("⚠ 本次未触发工具调用。可通过 MODEL_NAME 换用支持工具调用、能力更强的模型后重试。")

✓ 检测到工具调用，工具返回: echo: hello deepagents


## 4. 上层：用 `create_deep_agent()` 创建最小 Deep Agent

`create_deep_agent()` 来自 Deep Agents（Harness 层）。传入完全相同的模型和工具，但它会额外预置一套经过验证的工具——这正是第 1 章所说的「开箱即用的 Agent 套件」。

In [8]:
from deepagents import create_deep_agent

deep_agent = create_deep_agent(
    model=model,
    tools=[echo],
    system_prompt="你是一个简洁的助手。需要工具时优先调用工具，再用一句话总结结果。",
)

print("图节点:", list(deep_agent.get_graph().nodes))

图节点: ['__start__', 'model', 'tools', 'PatchToolCallsMiddleware.before_agent', '__end__']


### 4.1 对比两层默认装配的工具

读取编译后图中 `tools` 节点绑定的工具名，是观察「框架层 vs Harness 层」差异的确定手段——不依赖模型输出。

In [9]:
def list_tools(agent):
    """读取图中 tools 节点绑定的工具名，用于对比不同层默认装配的工具。"""
    node = agent.nodes["tools"]
    tools_by_name = getattr(node, "tools_by_name", None)
    if tools_by_name is None:
        bound = getattr(node, "bound", None)
        tools_by_name = getattr(bound, "tools_by_name", {})
    return sorted(tools_by_name)


lc_tools = list_tools(langchain_agent)
deep_tools = list_tools(deep_agent)

print("create_agent 暴露的工具       :", lc_tools)
print("create_deep_agent 暴露的工具  :", deep_tools)
print("Harness 额外提供的工具        :", [t for t in deep_tools if t not in lc_tools])

assert "write_file" in deep_tools, "Deep Agent 应默认提供虚拟文件系统工具"
print("✓ create_deep_agent 默认装配了虚拟文件系统工具（如 write_file / read_file / grep）")

create_agent 暴露的工具       : ['echo']
create_deep_agent 暴露的工具  : ['delete', 'echo', 'edit_file', 'execute', 'glob', 'grep', 'ls', 'read_file', 'task', 'write_file']
Harness 额外提供的工具        : ['delete', 'edit_file', 'execute', 'glob', 'grep', 'ls', 'read_file', 'task', 'write_file']
✓ create_deep_agent 默认装配了虚拟文件系统工具（如 write_file / read_file / grep）


### 4.2 用 Deep Agent 跑同样的任务

In [10]:
deep_result = deep_agent.invoke(
    {"messages": [{"role": "user", "content": question}]}
)

for message in deep_result["messages"]:
    message.pretty_print()

deep_tool_messages = [m for m in deep_result["messages"] if isinstance(m, ToolMessage)]
if deep_tool_messages:
    print("✓ Deep Agent 工具返回:", deep_tool_messages[-1].content)
else:
    print("⚠ 本次未触发工具调用。可通过 MODEL_NAME 换用能力更强的模型后重试。")

================================ Human Message =================================

请调用 echo 工具，把文本 'hello deepagents' 原样返回，然后告诉我工具返回了什么。
================================== Ai Message ==================================
Tool Calls:
  echo (01a0bdaed71814de36ee3d0fbc8b8446)
 Call ID: 01a0bdaed71814de36ee3d0fbc8b8446
  Args:
    text: hello deepagents
================================= Tool Message =================================
Name: echo

echo: hello deepagents
================================== Ai Message ==================================

工具返回了 "hello deepagents"。
✓ Deep Agent 工具返回: echo: hello deepagents


## 5. 三层架构在示例中的位置

| 层次 | 代表 | 本 Notebook 中的体现 |
|---|---|---|
| Runtime（底层） | LangGraph | `create_agent()` 和 `create_deep_agent()` 返回的是可运行的图；`get_graph().nodes` 里的 `model`、`tools` 节点就是 LangGraph 的执行引擎 |
| Framework（中间层） | LangChain | `from langchain.agents import create_agent`，以及统一的模型/工具接口（`ChatOpenAI`、`@tool`） |
| Harness（上层） | Deep Agents | `from deepagents import create_deep_agent`，在框架之上额外预置虚拟文件系统等工具 |

可以看到：传入相同的模型和工具时，`create_agent()` 只暴露你给的工具；`create_deep_agent()` 还会自动补齐 `write_file`、`read_file`、`grep`、`task` 等工具。这说明三层是**层层构建**而不是互相替代——Harness 复用了下面两层的运行时和框架能力。

## 预期现象

执行上面各步骤时，可以对照观察：

1. **消息轨迹**：`invoke` 返回的 `messages` 会依次出现
   `HumanMessage` →（`AIMessage`，带 `tool_calls`）→ `ToolMessage` → `AIMessage`。
   如果模型直接回答、没有调用工具，则不会出现 `ToolMessage`。
2. **工具返回**：触发工具调用时，`ToolMessage.content` 应为 `echo: hello deepagents`。
3. **两层工具差异**：`create_agent` 只暴露 `echo`；`create_deep_agent` 还会额外出现
   `ls`、`read_file`、`write_file`、`edit_file`、`delete`、`glob`、`grep`、`task` 等预置工具。
4. **确定性断言**：代码会校验「最后一条是 `AIMessage`」和「Deep Agent 含 `write_file`」，
   这两项与模型措辞无关；工具是否被调用则用提示信息而非固定回答来判定。


## 6. 常见报错、成本与清理

常见报错：

- `RuntimeError: 未检测到 SILICONFLOW_API_KEY`：按上文配置环境变量或 `.env` 后重跑；
- `AuthenticationError` / `401`：密钥错误或已失效，检查 Key 与 `SILICONFLOW_BASE_URL`；
- `APIConnectionError` / 超时：网络无法访问模型接口，检查网络或代理；
- 模型反复不调用工具：换用支持工具调用、能力更强的模型（通过 `MODEL_NAME` 指定）。

成本与网络：仅少量短对话，费用很低，但需要可访问模型接口的网络和有效账户。

资源清理：

- 本示例默认使用内存中的状态后端，不会在磁盘留下 Agent 写入的文件；
- 若你创建了 `.env`，它已在 `.gitignore` 中，请勿提交；
- 若要复现「干净环境」验证，可重启内核后从头运行本 Notebook。